

This notebook parses partner airline JSON redemption feeds into SQL Server tables and demonstrates how to join redemption data back to member profile data.


---

### Run each cell in order to execute the full pipeline.

In [6]:
# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------

import json
import logging
import pyodbc
from datetime import datetime

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------

SQL_SERVER = r"DESKTOP-ANIL"
SQL_DATABASE = "SkyPoints_DataEngineering"
SQL_DRIVER = "{ODBC Driver 18 for SQL Server}"

CONNECTION_STRING = (
    f"DRIVER={SQL_DRIVER};"
    f"SERVER={SQL_SERVER};"
    f"DATABASE={SQL_DATABASE};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()],
)

log = logging.getLogger(__name__)

print("Configuration loaded successfully.")

Configuration loaded successfully.


In [7]:
# ---------------------------------------------------------------------------
# SAMPLE REDEMPTION FEED DATA
# ---------------------------------------------------------------------------

# Sample redemption feed from partner airlines in JSON format
SAMPLE_REDEMPTION_FEED = '''
{
    "member_id": "MEM-001",
    "feed_date": "2024-01-15",
    "redemptions": [
        {
            "txn_id": "TXN-001",
            "txn_date": "2024-01-10",
            "partner": "Star Alliance",
            "miles_redeemed": 5000,
            "status": "completed"
        },
        {
            "txn_id": "TXN-002",
            "txn_date": "2024-01-12",
            "partner": "OneWorld",
            "miles_redeemed": 3000,
            "status": "completed"
        },
        {
            "txn_id": "TXN-003",
            "txn_date": "2024-01-14",
            "partner": "SkyTeam",
            "miles_redeemed": 7500,
            "status": "pending"
        }
    ]
}
'''

print(f"Sample redemption feed loaded with {len(json.loads(SAMPLE_REDEMPTION_FEED)['redemptions'])} records")

Sample redemption feed loaded with 3 records


In [8]:
# ---------------------------------------------------------------------------
# parse_feed_to_sql: Parse JSON redemption feed into SQL tables
# ---------------------------------------------------------------------------

def parse_feed_to_sql(feed_json_str, batch_id):
    """
    Parse a JSON redemption feed and insert records into the raw.Redemption_Raw
    table and the stg.Redemption_Stg staging table.
    """
    import pyodbc

    connection = pyodbc.connect(CONNECTION_STRING)
    cursor = connection.cursor()

    try:
        feed = json.loads(feed_json_str)
        member_id = feed.get("member_id")
        feed_date = feed.get("feed_date")
        redemptions = feed.get("redemptions", [])

        log.info(f"Parsing redemption feed for member_id={member_id}, feed_date={feed_date}, redemptions_count={len(redemptions)}")

        # Insert into raw table (full payload preservation)
        for idx, red in enumerate(redemptions):
            payload = json.dumps(red, ensure_ascii=False)
            cursor.execute("""
                INSERT INTO raw.Redemption_Raw
                (Source_File, Raw_Payload, File_Date, Batch_ID)
                VALUES (?, ?, ?, ?)
            """, (f"redemption_feed_{batch_id}", payload, feed_date, batch_id))

        # Insert into staging table (flattened)
        for red in redemptions:
            cursor.execute("""
                INSERT INTO stg.Redemption_Stg
                (Member_ID, Feed_Date, Transaction_ID, Transaction_Date,
                 Partner, Miles_Redeemed, Status, Source_File, Batch_ID)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                member_id,
                feed_date,
                red.get("txn_id"),
                red.get("txn_date"),
                red.get("partner"),
                red.get("miles_redeemed"),
                red.get("status"),
                f"redemption_feed_{batch_id}",
                batch_id,
            ))

        connection.commit()
        log.info(f"Redemption feed parsed successfully. Records inserted: {len(redemptions)}")

    except Exception as e:
        connection.rollback()
        log.error(f"Error parsing redemption feed: {e}")
        raise
    finally:
        cursor.close()
        connection.close()

print("parse_feed_to_sql function defined.")

parse_feed_to_sql function defined.


In [9]:
# ---------------------------------------------------------------------------
# join_redemption_to_member: Join redemption data to member profiles
# ---------------------------------------------------------------------------

def join_redemption_to_member():
    """
    Demonstrate joining redemption staging data to member profile data.
    """
    import pyodbc

    connection = pyodbc.connect(CONNECTION_STRING)
    cursor = connection.cursor()

    query = """
    SELECT
        m.Member_ID,
        m.Member_Name,
        m.Source_Country,
        r.Transaction_ID,
        r.Transaction_Date,
        r.Partner,
        r.Miles_Redeemed,
        r.Status,
        r.Feed_Date
    FROM stg.Redemption_Stg r
    INNER JOIN stg.Member_Profile_Stg m
        ON r.Member_ID = m.Member_ID
    ORDER BY m.Member_ID, r.Feed_Date DESC
    """

    cursor.execute(query)
    columns = [col[0] for col in cursor.description]
    results = cursor.fetchall()

    log.info(f"Join query returned {len(results)} rows")
    for row in results[:5]:  # Show first 5 as sample
        log.info(f"  {dict(zip(columns, row))}")

    cursor.close()
    connection.close()

print("join_redemption_to_member function defined.")

join_redemption_to_member function defined.


In [10]:
# ---------------------------------------------------------------------------
# main function
# ---------------------------------------------------------------------------

def main():
    log.info("=" * 60)
    log.info("INCUBYTE - PARSE REDEMPTION FEED")
    log.info("=" * 60)

    # 1. Parse sample feed into SQL
    batch_id = f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    parse_feed_to_sql(SAMPLE_REDEMPTION_FEED, batch_id)

    # 2. Demonstrate join to member profile
    log.info("Demonstrating redemption-to-member join:")
    join_redemption_to_member()

    log.info("=" * 60)
    log.info("REDEMPTION FEED PARSING COMPLETED")
    log.info("=" * 60)


if __name__ == "__main__":
    main()

2026-11-21 12:55:32,963 [INFO] ============================================================
2026-11-21 12:55:32,966 [INFO] INCUBYTE - PARSE REDEMPTION FEED
2026-11-21 12:55:32,968 [INFO] ============================================================
2026-11-21 12:55:32,973 [INFO] Parsing redemption feed for member_id=MEM-001, feed_date=2024-01-15, redemptions_count=3
2026-11-21 12:55:33,201 [INFO] Redemption feed parsed successfully. Records inserted: 3
2026-11-21 12:55:33,203 [INFO] Demonstrating redemption-to-member join:
2026-11-21 12:55:33,216 [INFO] Join query returned 0 rows
2026-11-21 12:55:33,217 [INFO] ============================================================
2026-11-21 12:55:33,220 [INFO] REDEMPTION FEED PARSING COMPLETED
2026-11-21 12:55:33,221 [INFO] ============================================================
